In [1]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("11) Random Forest/student_performance_dataset.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  1000 non-null   int64  
 1   gender                      1000 non-null   str    
 2   study_time_hours            1000 non-null   float64
 3   attendance_percent          1000 non-null   float64
 4   sleep_hours                 1000 non-null   float64
 5   parental_education          898 non-null    str    
 6   internet_access             1000 non-null   str    
 7   extracurricular_activities  1000 non-null   str    
 8   part_time_job               1000 non-null   str    
 9   previous_grade              1000 non-null   float64
 10  final_exam_score            1000 non-null   float64
 11  final_grade                 1000 non-null   str    
dtypes: float64(5), int64(1), str(6)
memory usage: 115.4 KB


In [2]:
df.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


In [3]:
df.isnull()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,True,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
995,False,False,False,False,False,False,False,False,False,False,False,False
996,False,False,False,False,False,True,False,False,False,False,False,False
997,False,False,False,False,False,False,False,False,False,False,False,False
998,False,False,False,False,False,False,False,False,False,False,False,False


In [4]:
df['parental_education'] = df['parental_education'].fillna('nessuna_istruzione')

In [5]:
df["parental_education"].value_counts()

parental_education
High School           356
Bachelors             308
Masters               184
nessuna_istruzione    102
PhD                    50
Name: count, dtype: int64

In [6]:
df = pd.get_dummies(df, columns=['gender', 'internet_access', 'extracurricular_activities', 'part_time_job'], drop_first=True)

In [7]:
valori = ['nessuna_istruzione', 'High School', 'Bachelors', 'Masters', 'PhD']
df["parental_education"] = df["parental_education"].map({v: i for i, v in enumerate(valori)})

In [8]:
df["final_grade"].value_counts()

final_grade
B    354
A    284
C    261
D     89
F     12
Name: count, dtype: int64

In [9]:
ordine_voti = ['F', 'D', 'C', 'B', 'A']
df['final_grade'] = df['final_grade'].map({v: i for i, v in enumerate(ordine_voti)})

In [10]:
df['final_grade'].value_counts()

final_grade
3    354
4    284
2    261
1     89
0     12
Name: count, dtype: int64

In [11]:
df.groupby('final_grade')['final_exam_score'].agg(['min', 'max'])

,min,max
final_grade,,
0,46.8,59.7
1,60.0,69.9
2,70.0,79.9
3,80.0,89.9
4,90.0,100.0


In [12]:
df['final_grade'] = df['final_grade'].replace({0: 1})

In [13]:
df['final_grade'].value_counts()

final_grade
3    354
4    284
2    261
1    101
Name: count, dtype: int64

In [14]:
X = df.drop(columns=['student_id', 'final_grade', 'final_exam_score'])
y = df['final_exam_score']

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   student_id                      1000 non-null   int64  
 1   study_time_hours                1000 non-null   float64
 2   attendance_percent              1000 non-null   float64
 3   sleep_hours                     1000 non-null   float64
 4   parental_education              1000 non-null   int64  
 5   previous_grade                  1000 non-null   float64
 6   final_exam_score                1000 non-null   float64
 7   final_grade                     1000 non-null   int64  
 8   gender_Male                     1000 non-null   bool   
 9   internet_access_Yes             1000 non-null   bool   
 10  extracurricular_activities_Yes  1000 non-null   bool   
 11  part_time_job_Yes               1000 non-null   bool   
dtypes: bool(4), float64(5), int64(3)
memory usage:

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [17]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

param_dist_forest = {
    'n_estimators': [100, 200, 500],
    'max_depth': [3, 5, 7],
    'max_features': ['sqrt', 'log2', None],
    'criterion': ['squared_error', 'absolute_error', 'poisson'],
}

param_dist_boosting = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'subsample': [0.6, 0.8, 1.0],
    'loss': ['squared_error', 'absolute_error', 'huber'],
}

random_search_forest = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_dist_forest, n_iter=20, cv=5, scoring='r2', random_state=42, error_score="raise"
)
random_search_forest.fit(X_train, y_train)

random_search_boosting = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_dist_boosting, n_iter=20, cv=5, scoring='r2', random_state=42, error_score="raise"
)
random_search_boosting.fit(X_train, y_train)

print(f"Random Forest - Migliori parametri: {random_search_forest.best_params_}")
print(f"Random Forest - Miglior R2 in CV: {random_search_forest.best_score_:.4f}")

print(f"Gradient Boosting - Migliori parametri: {random_search_boosting.best_params_}")
print(f"Gradient Boosting - Miglior R2 in CV: {random_search_boosting.best_score_:.4f}")

Random Forest - Migliori parametri: {'n_estimators': 200, 'max_features': 'sqrt', 'max_depth': 7, 'criterion': 'poisson'}
Random Forest - Miglior R2 in CV: 0.5906
Gradient Boosting - Migliori parametri: {'subsample': 0.8, 'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 3, 'loss': 'absolute_error'}
Gradient Boosting - Miglior R2 in CV: 0.6231


In [18]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

modelli = {
    "Random Forest": RandomForestRegressor(
        random_state=42,
        **random_search_forest.best_params_
    ),    
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
        **random_search_boosting.best_params_
    )
}

risultati = {}

for nome, modello in modelli.items():
    modello.fit(X_train, y_train)
    y_pred = modello.predict(X_test)
    
    risultati[nome] = {
        "R2": r2_score(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred)
    }

risultati_df = pd.DataFrame(risultati).T
print(risultati_df)

                         R2      RMSE       MAE
Random Forest      0.532673  7.053319  5.683691
Gradient Boosting  0.559873  6.844979  5.314712


In [19]:
rfg = modelli["Random Forest"]
gbc = modelli["Gradient Boosting"]
importanze_rfg = pd.Series(rfg.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importanze_gbc = pd.Series(gbc.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(f"Importanza Random Forest Regressor: {importanze_rfg}")
print(f"Importanza Gradient Boosting Regressor: {importanze_gbc}")

Importanza Random Forest Regressor: study_time_hours                  0.427017
previous_grade                    0.234134
attendance_percent                0.150683
sleep_hours                       0.071279
internet_access_Yes               0.041487
part_time_job_Yes                 0.032608
parental_education                0.023959
extracurricular_activities_Yes    0.010372
gender_Male                       0.008462
dtype: float64
Importanza Gradient Boosting Regressor: previous_grade                    0.293883
study_time_hours                  0.292517
attendance_percent                0.194681
sleep_hours                       0.109284
internet_access_Yes               0.040997
part_time_job_Yes                 0.030645
parental_education                0.020504
extracurricular_activities_Yes    0.011913
gender_Male                       0.005578
dtype: float64
